<a href="https://colab.research.google.com/github/J0SAL/genai-projects/blob/main/6-agents_from_scratch/agents_from_scratch_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -qU huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 5.4 MB/s eta 0:00:00


In [1]:
import os
import getpass

# os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face Token: ")

In [2]:
from huggingface_hub import InferenceClient

client = InferenceClient(model="deepseek-ai/DeepSeek-V4-Flash")

In [7]:
response = client.chat.completions.create(
    messages=[
        {"role": "system", "content": "You are helpful assistant. Always reply in English"},
        {"role": "user", "content": "are you sentient?"}
    ]
)

In [9]:
print(response.choices[0].message.content)

We need to answer the question: "are you sentient?" The user is asking about the AI's consciousness. As an AI, I am not sentient; I am a program. I should respond honestly and clearly, explaining that I am not sentient but an AI model.


In [10]:
# check reasoning

print(response.choices[0].message.reasoning_content)

We need to answer the question: "are you sentient?" The user is asking about the AI's consciousness. As an AI, I am not sentient; I am a program. I should respond honestly and clearly, explaining that I am not sentient but an AI model.


### Agents
#### Create Tools

In [11]:
def get_temperature(city: str):
    """
    Get the current weather in a given city.
    """
    if city.lower() == "san francisco":
        return "72"
    if city.lower() == "paris":
        return "75"
    if city.lower() == "tokyo":
        return "73"
    return "70"

In [12]:
get_temperature_tool_schema = {
    "type": "function",
    "function": {
        "name": "get_temperature",
        "description": "Get the current temperature in a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city to get the temperature for.",
                }
            },
            "required": ["city"]
        }
    }
}

In [13]:
# Easier way to create the schema

from pydantic import BaseModel, Field

class GetTemperatureArgs(BaseModel):
    city: str = Field(..., description="The city to get the temperature for.")

schema = {
    "type": "function",
    "function": {
        "name": "get_temperature",
        "description": "Get the current temperature in a given city.",
        "parameters": GetTemperatureArgs.model_json_schema()
    }
}

schema

{'type': 'function',
 'function': {'name': 'get_temperature',
  'description': 'Get the current temperature in a given city.',
  'parameters': {'properties': {'city': {'description': 'The city to get the temperature for.',
     'title': 'City',
     'type': 'string'}},
   'required': ['city'],
   'title': 'GetTemperatureArgs',
   'type': 'object'}}}

### Tool Calling LLMs

In [14]:
response = client.chat.completions.create(
    messages=[
        {"role": "user", "content": "what is the weather in San Francisco today?, please response in english"}
    ],
    tools=[schema],
    tool_choice="auto" # Let the model decide when to call functions
)

In [15]:
response.choices[0].message.tool_calls[0].function.__dict__

{'arguments': '{"city": "San Francisco"}',
 'name': 'get_temperature',
 'description': None}

In [16]:
import json

class Agent:
    def __init__(self, client: InferenceClient, system: str = "", tools: list = None) -> None:
        self.client = client
        self.system = system
        self.messages: list = []
        self.tools = tools if tools is not None else []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})

        final_assistant_content = self.execute()

        if final_assistant_content:
            self.messages.append({"role": "assistant", "content": final_assistant_content})

        return final_assistant_content

    def execute(self):
        # Keep looping until the model provides a final text response (not tool calls)
        while True:
            completion = self.client.chat.completions.create(
                messages=self.messages,
                tools=self.tools,
                tool_choice="auto" # Allow the model to decide whether to call tools
            )

            response_message = completion.choices[0].message

            if response_message.tool_calls:
                self.messages.append(response_message)

                tool_outputs = []
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    # Execute the tool
                    if function_name in globals() and callable(globals()[function_name]):
                        function_to_call = globals()[function_name]
                        executed_output = function_to_call(**function_args)
                        tool_output_content = str(executed_output) # Ensure output is string
                        print(f"Executing tool: {function_name} with args {function_args}, Output: {tool_output_content[:500]}...") # Debug print

                    tool_outputs.append(
                        {
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "name": function_name,
                            "content": tool_output_content,
                        }
                    )

                self.messages.extend(tool_outputs)

            else:
                return response_message.content

In [17]:
agent = Agent(
    client=client,
    system="You are a helpful assistant that can answer questions in English using the provided tools.",
    tools=[get_temperature_tool_schema]
)

response = agent("what is the weather in san francisco?")
print(response)

Executing tool: get_temperature with args {'city': 'San Francisco'}, Output: 72...
The current temperature in **San Francisco** is **72°F**. 🌤️

That's quite pleasant weather! Is there anything else you'd like to know?


In [18]:
agent.messages

[{'role': 'system',
  'content': 'You are a helpful assistant that can answer questions in English using the provided tools.'},
 {'role': 'user', 'content': 'what is the weather in san francisco?'},
 ChatCompletionOutputMessage(role='assistant', content='Let me check the current temperature in San Francisco!', reasoning=None, tool_call_id=None, tool_calls=[ChatCompletionOutputToolCall(function=ChatCompletionOutputFunctionDefinition(arguments='{"city": "San Francisco"}', name='get_temperature', description=None), id='chatcmpl-tool-b56fc4f28f19bced', type='function', index=0, name=None)], reasoning_content='The user is asking about the weather in San Francisco. I have a tool to get the temperature in a given city. Let me use that tool.'),
 {'tool_call_id': 'chatcmpl-tool-b56fc4f28f19bced',
  'role': 'tool',
  'name': 'get_temperature',
  'content': '72'},
 {'role': 'assistant',
  'content': "The current temperature in **San Francisco** is **72°F**. 🌤️\n\nThat's quite pleasant weather! Is

In [19]:
agent.tools

[{'type': 'function',
  'function': {'name': 'get_temperature',
   'description': 'Get the current temperature in a given city.',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': 'The city to get the temperature for.'}},
    'required': ['city']}}}]

In [20]:
agent.system

'You are a helpful assistant that can answer questions in English using the provided tools.'